In [4]:
import pandas as pd

In [15]:
audit=pd.read_csv(r"C:\Users\chaba\Documents\AdvDip\IDA117V\IDA117V-Audit-Risk-Project\audit_opinion_facts_2026-07-30_11-42-36.csv")
incexp=pd.read_csv(r"C:\Users\chaba\Documents\AdvDip\IDA117V\IDA117V-Audit-Risk-Project\incexp_v2_aggregate_2026-09-27T17_44_27.115803.csv")
cflow=pd.read_csv(r"C:\Users\chaba\Documents\AdvDip\IDA117V\IDA117V-Audit-Risk-Project\cflow_v2_aggregate_2026-09-27T17_44_51.065924.csv")
creditor =pd.read_csv(r"C:\Users\chaba\Documents\AdvDip\IDA117V\IDA117V-Audit-Risk-Project\aged_creditor_v2_aggregate_2026-09-27T17_45_23.500582.csv")
finpos =pd.read_csv(r"C:\Users\chaba\Documents\AdvDip\IDA117V\IDA117V-Audit-Risk-Project\financial_position_v2_aggregate_2026-09-27T18_58_54.778216.csv")

In [16]:
for name, d in [("audit", audit), ("incexp", incexp), ("cflow", cflow),
                ("creditor", creditor), ("finpos", finpos)]:
    print(name, "->", d.columns.tolist())

audit -> ['Demarcation Code', 'Name', 'Year End', 'Code', 'Label', "Auditor General's Report URL"]
incexp -> ['amount_type.label', 'amount_type.code', 'financial_period.period', 'period_length.length', 'financial_year_end.year', 'item.composition', 'item.return_form_structure', 'item.position_in_return_form', 'item.label', 'item.code', 'demarcation.label', 'demarcation.code', 'amount.sum']
cflow -> ['amount_type.label', 'amount_type.code', 'financial_period.period', 'period_length.length', 'financial_year_end.year', 'item.composition', 'item.return_form_structure', 'item.position_in_return_form', 'item.label', 'item.code', 'demarcation.label', 'demarcation.code', 'amount.sum']
creditor -> ['amount_type.label', 'amount_type.code', 'financial_period.period', 'period_length.length', 'financial_year_end.year', 'item.composition', 'item.return_form_structure', 'item.position_in_return_form', 'item.label', 'item.code', 'demarcation.label', 'demarcation.code', 'g1_amount.sum', 'l1_amount.sum'

In [17]:
incexp["amount_type.label"].value_counts()

amount_type.label
Audited Actual    331
Name: count, dtype: int64

The dataset contains 3688 municipality-year records and 6 variables

In [18]:
print(incexp.shape)
print(incexp["demarcation.code"].nunique())
print(sorted(incexp["financial_year_end.year"].unique()))


(331, 13)
8
[np.int64(2024)]


Identifies which variables are categorical and numeric

In [21]:
"""
Municipal Audit Risk Project — full data pull + merge script.
Run this on your own machine (not in a sandbox) since it needs internet access.

What it does:
1. Pulls audit_opinions (your target) + 4 financial cubes (your features) via the API,
   paging through until every row is collected.
2. Reshapes the financial cubes from "long" format (one row per line item) into
   "wide" format (one row per municipality-year, one column per item).
3. Merges everything into one master_dataset.csv, ready for EDA/modelling.
"""

import requests
import pandas as pd
import time

BASE = "https://municipaldata.treasury.gov.za/api/cubes"

# Cube names confirmed from the API's own /api/cubes listing
CUBES = {
    "audit": "audit_opinions",
    "incexp": "incexp_v2",
    "cflow": "cflow_v2",
    "creditor": "aged_creditor_v2",
    "finpos": "financial_position_v2",
}


def fetch_all(cube_name, page_size=10000, max_pages=200):
    """Pull every row from a cube by paging through the API."""
    all_rows = []
    page = 0
    while page < max_pages:
        url = f"{BASE}/{cube_name}/facts"
        params = {"pagesize": page_size, "page": page}
        try:
            r = requests.get(url, params=params, timeout=90)
        except requests.exceptions.RequestException as e:
            print(f"  [{cube_name}] request failed on page {page}: {e}")
            break

        if r.status_code != 200:
            print(f"  [{cube_name}] HTTP {r.status_code} on page {page}: {r.text[:300]}")
            break

        payload = r.json()
        batch = payload.get("data", [])
        if not batch:
            break

        all_rows.extend(batch)
        print(f"  [{cube_name}] page {page}: +{len(batch)} rows (total {len(all_rows)})")
        page += 1
        time.sleep(0.2)  # be polite to the API

    return pd.DataFrame(all_rows)


def find_value_column(df):
    """Auto-detect the numeric 'amount' column, since its exact name can vary."""
    candidates = [c for c in df.columns if "amount" in c.lower()]
    # Prefer something ending in .sum, else just take the first amount-like column
    for c in candidates:
        if c.lower().endswith(".sum"):
            return c
    return candidates[0] if candidates else None


def pivot_long_to_wide(df, index_cols, item_col, value_col=None):
    """Turn one-row-per-line-item data into one-row-per-municipality-year."""
    if df.empty:
        print("  (empty dataframe, skipping pivot)")
        return df

    print(f"  columns available: {df.columns.tolist()}")

    if value_col is None or value_col not in df.columns:
        detected = find_value_column(df)
        if detected is None:
            print("  WARNING: no amount-like column found, skipping pivot")
            return pd.DataFrame(columns=index_cols)
        print(f"  using detected value column: '{detected}' (expected '{value_col}')")
        value_col = detected

    missing_index = [c for c in index_cols if c not in df.columns]
    if missing_index:
        print(f"  WARNING: missing index columns {missing_index}, skipping pivot")
        return pd.DataFrame(columns=index_cols)

    wide = df.pivot_table(
        index=index_cols,
        columns=item_col,
        values=value_col,
        aggfunc="sum",
    ).reset_index()
    wide.columns = [str(c) for c in wide.columns]
    return wide


def main():
    print("Downloading all cubes (this can take a few minutes)...\n")

    raw = {}
    for key, cube in CUBES.items():
        print(f"Fetching {cube} ...")
        raw[key] = fetch_all(cube)
        print(f"-> {key}: {raw[key].shape}\n")
        raw[key].to_csv(f"data_raw_{key}.csv", index=False)

    audit = raw["audit"]

    # Standardise join keys on the audit table
    audit = audit.rename(columns={
        "Demarcation Code": "demarcation.code",
        "Year End": "financial_year_end.year",
    })

    index_cols = ["demarcation.code", "financial_year_end.year"]

    print("Pivoting financial cubes into wide format...")
    incexp_wide = pivot_long_to_wide(raw["incexp"], index_cols, "item.label", "amount.sum")
    cflow_wide = pivot_long_to_wide(raw["cflow"], index_cols, "item.label", "amount.sum")
    finpos_wide = pivot_long_to_wide(raw["finpos"], index_cols, "item.label", "amount.sum")

    # aged_creditor_v2 has multiple amount columns (l30, l60, l90 ... total)
    creditor_value_cols = [c for c in raw["creditor"].columns if c.endswith("_amount.sum")]
    if not creditor_value_cols:
        creditor_value_cols = [c for c in raw["creditor"].columns if "amount" in c.lower()]
    print(f"  creditor value columns detected: {creditor_value_cols}")
    if not raw["creditor"].empty and creditor_value_cols:
        creditor_wide = raw["creditor"].pivot_table(
            index=index_cols,
            columns="item.label",
            values=creditor_value_cols,
            aggfunc="sum",
        )
        creditor_wide.columns = ["_".join(map(str, c)) for c in creditor_wide.columns]
        creditor_wide = creditor_wide.reset_index()
    else:
        creditor_wide = pd.DataFrame(columns=index_cols)

    print("Merging everything into one master dataset...")
    merged = audit.merge(incexp_wide, on=index_cols, how="left") \
                  .merge(cflow_wide, on=index_cols, how="left") \
                  .merge(finpos_wide, on=index_cols, how="left") \
                  .merge(creditor_wide, on=index_cols, how="left")

    print(f"\nFinal merged shape: {merged.shape}")
    merged.to_csv("master_dataset.csv", index=False)
    print("Saved: master_dataset.csv")


if __name__ == "__main__":
    main()


Fetching audit_opinions ...
  [audit_opinions] page 0: +3688 rows (total 3688)
  [audit_opinions] page 1: +3688 rows (total 7376)
-> audit: (7376, 6)

Fetching incexp_v2 ...
  [incexp_v2] page 0: +10000 rows (total 10000)
  [incexp_v2] page 1: +10000 rows (total 20000)
  [incexp_v2] page 2: +10000 rows (total 30000)
  [incexp_v2] page 3: +10000 rows (total 40000)
  [incexp_v2] page 4: +10000 rows (total 50000)
  [incexp_v2] page 5: +10000 rows (total 60000)
  [incexp_v2] page 6: +10000 rows (total 70000)
  [incexp_v2] page 7: +10000 rows (total 80000)
  [incexp_v2] page 8: +10000 rows (total 90000)
  [incexp_v2] page 9: +10000 rows (total 100000)
  [incexp_v2] page 10: +10000 rows (total 110000)
  [incexp_v2] page 11: +10000 rows (total 120000)
  [incexp_v2] page 12: +10000 rows (total 130000)
  [incexp_v2] page 13: +10000 rows (total 140000)
  [incexp_v2] page 14: +10000 rows (total 150000)
  [incexp_v2] HTTP 504 on page 15: <!DOCTYPE html>
<html>
<head>
  <title>We're sorry, but som

In [24]:
"""
Run this AFTER you have `merged` (your master dataset) in memory, or load it from CSV.
Produces:
  - data_dictionary.csv   -> one row per column, for your Q2.1 report section
  - audit_summary.txt     -> missingness, duplicates, target balance, for the same section
"""

import pandas as pd

merged = pd.read_csv("master_dataset.csv")  # skip this line if already in memory

# ---- 1. Data dictionary ----
rows = []
for col in merged.columns:
    non_null = merged[col].notna().sum()
    total = len(merged)
    rows.append({
        "column": col,
        "dtype": str(merged[col].dtype),
        "non_null_count": non_null,
        "missing_pct": round(100 * (1 - non_null / total), 1),
        "n_unique": merged[col].nunique(),
        "example_value": merged[col].dropna().iloc[0] if non_null > 0 else None,
    })

data_dict = pd.DataFrame(rows)
data_dict.to_csv("data_dictionary.csv", index=False)
print(f"Saved data_dictionary.csv with {len(data_dict)} columns documented")

# ---- 2. Audit summary ----
with open("audit_summary.txt", "w", encoding="utf-8") as f:
    f.write(f"Shape: {merged.shape}\n\n")

    dup_count = merged.duplicated(subset=["demarcation.code", "financial_year_end.year"]).sum()
    f.write(f"Duplicate municipality-year rows: {dup_count}\n\n")

    if "Label" in merged.columns:
        f.write("Target class balance (Label):\n")
        f.write(merged["Label"].value_counts().to_string())
        f.write("\n\n")

    f.write("Top 15 columns by missingness:\n")
    missing = data_dict.sort_values("missing_pct", ascending=False).head(15)
    f.write(missing[["column", "missing_pct"]].to_string(index=False))
    f.write("\n")



C:\Users\chaba\AppData\Local\Temp\ipykernel_47496\1003312579.py:10: DtypeWarning: Columns (6,7,8,9,10,11,12,13,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,44,45,46,47,48,49,50,51,52,53,54,55) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv("master_dataset.csv")  # skip this line if already in memory


Saved data_dictionary.csv with 233 columns documented


In [25]:
index_cols = ["demarcation.code", "financial_year_end.year"]

# rename audit's join keys to match the others
audit_r = audit.rename(columns={
    "Demarcation Code": "demarcation.code",
    "Year End": "financial_year_end.year",
})

def pivot_wide(df, value_col):
    return df.pivot_table(
        index=index_cols, columns="item.label", values=value_col, aggfunc="sum"
    ).reset_index()

incexp_wide = pivot_wide(incexp, "amount.sum")
cflow_wide = pivot_wide(cflow, "amount.sum")
finpos_wide = pivot_wide(finpos, "amount.sum")

creditor_cols = [c for c in creditor.columns if "amount" in c.lower() and c not in index_cols]
creditor_wide = creditor.pivot_table(
    index=index_cols, columns="item.label", values=creditor_cols, aggfunc="sum"
)
creditor_wide.columns = ["_".join(map(str, c)) for c in creditor_wide.columns]
creditor_wide = creditor_wide.reset_index()

# prefix every column (except the join keys) so nothing can collide across tables
def prefix(df, name):
    return df.rename(columns={c: f"{name}__{c}" for c in df.columns if c not in index_cols})

incexp_wide = prefix(incexp_wide, "incexp")
cflow_wide = prefix(cflow_wide, "cflow")
finpos_wide = prefix(finpos_wide, "finpos")
creditor_wide = prefix(creditor_wide, "creditor")

# THE KEY CHECK: is each table already unique per municipality-year, before we merge anything?
for name, w in [("audit", audit_r), ("incexp_wide", incexp_wide), ("cflow_wide", cflow_wide),
                ("finpos_wide", finpos_wide), ("creditor_wide", creditor_wide)]:
    dup = w.duplicated(subset=index_cols).sum()
    print(name, w.shape, "duplicate keys:", dup)

audit (3688, 6) duplicate keys: 0
incexp_wide (8, 54) duplicate keys: 0
cflow_wide (8, 22) duplicate keys: 0
finpos_wide (8, 37) duplicate keys: 0
creditor_wide (7, 112) duplicate keys: 0


In [27]:
import requests

url = "https://municipaldata.treasury.gov.za/api/cubes/incexp_v2/aggregate"
params = {
    "drilldown": "demarcation.code|demarcation.label|financial_year_end.year|"
                 "item.code|item.label|amount_type.label|financial_period.period|period_length.length",
    "aggregates": "amount.sum",
    "cut": "financial_year_end.year:2024",
    "pagesize": 100,
}

r = requests.get(url, params=params, timeout=90)
print("STATUS:", r.status_code)
print(r.text[:800])

STATUS: 200
{"total_cell_count": 96272, "cells": [{"demarcation.code": "BUF", "demarcation.label": "Buffalo City", "financial_year_end.year": 2024, "item.code": "0300", "item.label": "Service charges - Electricity", "amount_type.label": "Actual", "financial_period.period": 1, "period_length.length": "month", "amount.sum": 196903320.0}, {"demarcation.code": "BUF", "demarcation.label": "Buffalo City", "financial_year_end.year": 2024, "item.code": "0300", "item.label": "Service charges - Electricity", "amount_type.label": "Actual", "financial_period.period": 2, "period_length.length": "month", "amount.sum": 203961884.0}, {"demarcation.code": "BUF", "demarcation.label": "Buffalo City", "financial_year_end.year": 2024, "item.code": "0300", "item.label": "Service charges - Electricity", "amount_type.label":


In [29]:
import requests
import pandas as pd

url = "https://municipaldata.treasury.gov.za/api/cubes/incexp_v2/aggregate"
params = {
    "drilldown": "amount_type.label|period_length.length|financial_year_end.year",
    "aggregates": "amount.sum",
    "pagesize": 1000,
}

r = requests.get(url, params=params, timeout=90)
print("STATUS:", r.status_code)
df = pd.DataFrame(r.json()["cells"])
print(df.groupby(["amount_type.label", "period_length.length"])["financial_year_end.year"]
        .apply(lambda s: sorted(s.unique())).to_string())

STATUS: 200
amount_type.label                      period_length.length
Actual                                 month                               [2020, 2021, 2022, 2023, 2024, 2026]
Adjusted Budget                        year                                [2020, 2021, 2022, 2023, 2024, 2026]
Audited Actual                         year                    [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Forecast 1 year ahead of budget year   year                                            [2021, 2022, 2023, 2025]
Forecast 2 years ahead of budget year  year                                            [2021, 2022, 2023, 2025]
ITY1                                   year                                            [2021, 2022, 2023, 2025]
ITY2                                   year                                            [2021, 2022, 2023, 2025]
Original Budget                        year                          [2020, 2021, 2022, 2023, 2024, 2025, 2026]
Pre-audit                       

In [57]:
import requests
import pandas as pd

BASE = "https://municipaldata.treasury.gov.za/api/cubes"

def fetch_aggregate(cube, drilldown, aggregates, cut, pagesize=10000):
    """Pull every page of an /aggregate query."""
    rows, page = [], 0
    while True:
        params = {
            "drilldown": drilldown,
            "aggregates": aggregates,
            "cut": cut,
            "pagesize": pagesize,
            "page": page,
        }
        r = requests.get(f"{BASE}/{cube}/aggregate", params=params, timeout=120)
        if r.status_code != 200:
            print("ERROR", r.status_code, r.text[:500])
            break
        data = r.json()
        rows.extend(data["cells"])
        print(f"{cube} year cut: fetched {len(rows)} of {data['total_cell_count']}")
        if len(rows) >= data["total_cell_count"] or not data["cells"]:
            break
        page += 1
    return pd.DataFrame(rows)

DRILL = "demarcation.code|demarcation.label|financial_year_end.year|item.code|item.label"

frames = []
for year in range(2018, 2026):          # Pre-audit exists for 2018-2025
    cut = f'financial_year_end.year:{year}|amount_type.label:"Pre-audit"'
    frames.append(fetch_aggregate("incexp_v2", DRILL, "amount.sum", cut))

incexp = pd.concat(frames, ignore_index=True)
print(incexp.shape)
print(incexp["demarcation.code"].nunique(), "municipalities")
print(sorted(incexp["financial_year_end.year"].unique()))
incexp.to_csv("incexp_v2_preaudit_all.csv", index=False)

incexp_v2 year cut: fetched 5637 of 5637
incexp_v2 year cut: fetched 5857 of 5857
incexp_v2 year cut: fetched 6166 of 6166
incexp_v2 year cut: fetched 6406 of 6406
incexp_v2 year cut: fetched 6451 of 6451
incexp_v2 year cut: fetched 8278 of 8278
incexp_v2 year cut: fetched 8528 of 8528
incexp_v2 year cut: fetched 8666 of 8666
(55989, 6)
257 municipalities
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [58]:
CUBES = {
    "cflow": ("cflow_v2", "amount.sum"),
    "finpos": ("financial_position_v2", "amount.sum"),
    "creditor": ("aged_creditor_v2",
                 "g1_amount.sum|l1_amount.sum|l30_amount.sum|l60_amount.sum|"
                 "l90_amount.sum|l120_amount.sum|l150_amount.sum|l180_amount.sum|total_amount.sum"),
}

data = {"incexp": incexp}
for key, (cube, aggs) in CUBES.items():
    frames = []
    for year in range(2018, 2026):
        cut = f'financial_year_end.year:{year}|amount_type.label:"Pre-audit"'
        frames.append(fetch_aggregate(cube, DRILL, aggs, cut))
    df = pd.concat(frames, ignore_index=True)
    if df.empty:
        print(f"WARNING: no Pre-audit rows for {cube}")
    else:
        print(f"{key}: {df.shape}, {df['demarcation.code'].nunique()} municipalities")
    df.to_csv(f"{cube}_preaudit_all.csv", index=False)
    data[key] = df

# ---- reshape each cube: one row per municipality-year, one column per item ----
idx = ["demarcation.code", "financial_year_end.year"]

def to_wide(df, prefix):
    df = df.copy()
    df["feature"] = df["item.code"].astype(str) + "_" + df["item.label"]
    value_cols = [c for c in df.columns if c.endswith(".sum")]
    wide = df.pivot_table(index=idx, columns="feature", values=value_cols, aggfunc="sum")
    wide.columns = [f"{prefix}__{feat}" if len(value_cols) == 1
                    else f"{prefix}__{feat}__{val.replace('.sum','')}"
                    for val, feat in wide.columns]
    return wide.reset_index()

wides = {k: to_wide(v, k) for k, v in data.items() if not v.empty}

# ---- merge onto the audit target ----
audit_r = audit.rename(columns={"Demarcation Code": "demarcation.code",
                                "Year End": "financial_year_end.year"})
audit_r = audit_r[audit_r["financial_year_end.year"].between(2018, 2025)]

master = audit_r
for k, w in wides.items():
    master = master.merge(w, on=idx, how="left")

print(master.shape)
print("duplicate keys:", master.duplicated(subset=idx).sum())
print(master["Code"].value_counts())
master.to_csv("master_dataset.csv", index=False)

cflow_v2 year cut: fetched 586 of 586
cflow_v2 year cut: fetched 1056 of 1056
cflow_v2 year cut: fetched 1256 of 1256
cflow_v2 year cut: fetched 1796 of 1796
cflow_v2 year cut: fetched 2116 of 2116
cflow_v2 year cut: fetched 2312 of 2312
cflow_v2 year cut: fetched 2476 of 2476
cflow_v2 year cut: fetched 2502 of 2502
cflow: (14100, 6), 257 municipalities
financial_position_v2 year cut: fetched 6060 of 6060
financial_position_v2 year cut: fetched 6125 of 6125
financial_position_v2 year cut: fetched 6634 of 6634
financial_position_v2 year cut: fetched 4846 of 4846
financial_position_v2 year cut: fetched 4860 of 4860
financial_position_v2 year cut: fetched 6997 of 6997
financial_position_v2 year cut: fetched 7174 of 7174
financial_position_v2 year cut: fetched 7220 of 7220
finpos: (49916, 6), 257 municipalities
aged_creditor_v2 year cut: fetched 0 of 0
aged_creditor_v2 year cut: fetched 0 of 0
aged_creditor_v2 year cut: fetched 1900 of 1900
aged_creditor_v2 year cut: fetched 2380 of 2380
a

In [59]:
master = pd.read_csv("master_dataset.csv", low_memory=False)
print(master.groupby("financial_year_end.year").size())
print(master.groupby("financial_year_end.year")["Code"].apply(lambda s: (s == "outstanding").sum()))
feat = [c for c in master.columns if "__" in c]
print(master.groupby("financial_year_end.year")[feat].apply(lambda d: d.isna().mean().mean()).round(2))

financial_year_end.year
2018    257
2019    257
2020    257
2021    257
2022    257
2023    257
2024    257
dtype: int64
financial_year_end.year
2018     4
2019    28
2020     1
2021    24
2022     2
2023    10
2024    10
Name: Code, dtype: int64
financial_year_end.year
2018    0.78
2019    0.76
2020    0.43
2021    0.37
2022    0.37
2023    0.28
2024    0.27
dtype: float64
